In [25]:
!curl -L -o ag-news-classification-dataset.zip\
  https://www.kaggle.com/api/v1/datasets/download/amananandrai/ag-news-classification-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 11.3M  100 11.3M    0     0  37.9M      0 --:--:-- --:--:-- --:--:-- 37.9M


In [26]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_device(device)

In [27]:
!unzip /content/ag-news-classification-dataset.zip

Archive:  /content/ag-news-classification-dataset.zip
replace test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: test.csv                
  inflating: train.csv               


In [28]:
import pandas as pd

test_df = pd.read_csv('/content/test.csv')
train_df = pd.read_csv('/content/train.csv')

In [29]:
test_df

,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...
...,...,...,...
7595,1,Around the world,Ukrainian presidential candidate Viktor Yushch...
7596,2,Void is filled with Clement,With the supply of attractive pitching options...
7597,2,Martinez leaves bitter,Like Roger Clemens did almost exactly eight ye...
7598,3,5 of arthritis patients in Singapore take Bext...,SINGAPORE : Doctors in the United States have ...


In [30]:
test_df['Class Index'].value_counts()

,count
Class Index,
3,1900
4,1900
2,1900
1,1900


# Tokenization

In [31]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import ByteLevel

In [32]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

In [33]:
special_tokens = [
    '<PAD>',
    '<UNK>'
]


In [34]:
trainer = BpeTrainer(
    vocab_size = 16000,
    special_tokens = special_tokens,
    show_progress = True
)

In [35]:
def get_training_corpus(batch_size = 1000):
  train_df = pd.read_csv('/content/train.csv')

  for start in range(0, len(train_df), batch_size):
    desc = train_df.iloc[start:start+batch_size]['Description'].values
    cls = train_df.iloc[start:start+batch_size]['Class Index'].astype(str).values
    yield desc + cls

In [36]:
tokenizer.train_from_iterator(
    get_training_corpus(),
    trainer = trainer
)

In [37]:
tokenizer.get_vocab_size()

16000

In [38]:
example_text = 'Hi how are you?'
encoded = tokenizer.encode(example_text)
print(encoded.tokens)
print(encoded.ids)

['H', 'i', 'how', 'are', 'you', '?']
[36, 65, 1294, 186, 839, 28]


In [39]:
PAD_ID = tokenizer.token_to_id('<PAD>')
UNK_ID = tokenizer.token_to_id('<UNK>')

In [40]:
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

In [41]:
len(train_df), len(test_df)

(120000, 7600)

In [42]:
train_df, valid_df = train_df.iloc[:112400], train_df.iloc[112400:]

In [43]:
len(train_df), len(valid_df), len(test_df)

(112400, 7600, 7600)

In [44]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

In [45]:
class AGNewsDataset(Dataset):
  def __init__(self, df, tokenizer):
    self.df = df
    self.tokenizer = tokenizer

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    encoded = self.tokenizer.encode(self.df.iloc[idx]['Description'])
    label = self.df.iloc[idx]['Class Index'] - 1

    encoded = torch.tensor(encoded.ids, dtype = torch.long)
    label = torch.tensor(label, dtype = torch.long)

    return encoded, label.unsqueeze(0)

In [46]:
def collate_fn(batch):
    sequences, labels = zip(*batch)
    padded_sequences = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)
    return padded_sequences, torch.stack(labels)

torch.set_default_device(None)

train_dataset = AGNewsDataset(train_df, tokenizer)
valid_dataset = AGNewsDataset(valid_df, tokenizer)
test_dataset = AGNewsDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

try:
    batch_x, batch_y = next(iter(train_loader))
    print(f"Batch X shape: {batch_x.shape}")
except Exception as e:
    print(f"Error: {e}")

Batch X shape: torch.Size([32, 67])


In [47]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

In [48]:
import math

class AGNewsClassification(nn.Module):
  def __init__(self, vocab_size, nheads, embedding_dim, ff_hidden_dim, num_classes, num_layers = 5, dropout = 0.1):
    super(AGNewsClassification, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.pos_encoder = PositionalEncoding(embedding_dim, dropout)
    self.dropout = nn.Dropout(p=dropout)

    self.encoder = nn.TransformerEncoder(
        nn.TransformerEncoderLayer(
            d_model = embedding_dim,
            nhead = nheads,
            dim_feedforward = ff_hidden_dim,
            dropout = dropout,
            batch_first = True
        ),
        num_layers = num_layers
    )

    self.classifier = nn.Linear(embedding_dim, num_classes)

  def forward(self, x):
    x = self.embedding(x)
    # Use the pre-initialized positional encoder
    x = self.pos_encoder(x)
    x = self.encoder(x)
    x = x.mean(dim = 1)
    x = self.dropout(x)
    x = self.classifier(x)
    return x

# Training

In [51]:
import tqdm

def train_one_epoch(model, optimizer, criterion, train_loader, device):
  model.train()
  total_loss = 0.0
  total_correct = 0
  total_samples = 0

  pbar = tqdm.tqdm(train_loader, desc="Training")

  for inputs, targets in pbar:
    inputs, targets = inputs.to(device), targets.to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets.squeeze(1))
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    total_correct += (predicted == targets.squeeze(1)).sum().item()
    total_samples += targets.size(0)

    current_acc = total_correct / total_samples
    pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{current_acc:.4f}"})

  avg_loss = total_loss / len(train_loader)
  accuracy = total_correct / total_samples

  return avg_loss, accuracy

@torch.no_grad
def evaluate(model, criterion, loader, device):
  model.eval()
  total_loss = 0.0
  total_correct = 0
  total_samples = 0

  pbar = tqdm.tqdm(loader, desc="Evaluating")

  for inputs, targets in pbar:
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)
    loss = criterion(outputs, targets.squeeze(1))

    total_loss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    total_correct += (predicted == targets.squeeze(1)).sum().item()
    total_samples += targets.size(0)

    current_acc = total_correct / total_samples
    pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{current_acc:.4f}"})

  avg_loss = total_loss / len(loader)
  accuracy = total_correct / total_samples

  return avg_loss, accuracy

def train(model, optimizer, criterion, train_loader, valid_loader, epochs, device):
  for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, optimizer, criterion, train_loader, device)
    valid_loss, valid_acc = evaluate(model, criterion, valid_loader, device)
    print(f"Epoch: {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.4f}")

In [53]:
import sys
import importlib
import torch
import torch.nn as nn
import math
import sympy

# Datasets
train_dataset = AGNewsDataset(train_df, tokenizer)
valid_dataset = AGNewsDataset(valid_df, tokenizer)
test_dataset = AGNewsDataset(test_df, tokenizer)

# Loaders (CPU based creation)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AGNewsClassification(
    vocab_size = tokenizer.get_vocab_size(),
    nheads = 8,
    embedding_dim = 256,
    ff_hidden_dim = 512,
    num_classes = 4,
    num_layers = 3
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
criterion = nn.CrossEntropyLoss()

print(f"Training starts, device: {device}")
train(model, optimizer, criterion, train_loader, valid_loader, 5, device)

Training starts, device: cuda


Evaluating: 100%|██████████| 238/238 [00:03<00:00, 67.81it/s, loss=0.1130, acc=0.8575]


Epoch: 1/5, Train Loss: 0.6075, Train Acc: 0.7592, Valid Loss: 0.3937, Valid Acc: 0.8575


Evaluating: 100%|██████████| 238/238 [00:03<00:00, 62.66it/s, loss=0.0450, acc=0.8767]


Epoch: 2/5, Train Loss: 0.3727, Train Acc: 0.8657, Valid Loss: 0.3399, Valid Acc: 0.8767


...

Epoch: 3/5, Train Loss: 0.3190, Train Acc: 0.8872, Valid Loss: 0.3141, Valid Acc: 0.8879


Evaluating: 100%|██████████| 238/238 [00:03<00:00, 59.87it/s, loss=0.0139, acc=0.8911]


Epoch: 4/5, Train Loss: 0.2832, Train Acc: 0.8980, Valid Loss: 0.3009, Valid Acc: 0.8911


Evaluating: 100%|██████████| 238/238 [00:04<00:00, 58.11it/s, loss=0.0784, acc=0.8963]

Epoch: 5/5, Train Loss: 0.2589, Train Acc: 0.9074, Valid Loss: 0.2916, Valid Acc: 0.8963


In [54]:
test_loss, test_acc = evaluate(model, criterion, test_loader, device)
print(f"\nFinal test results:\nLoss: {test_loss:.4f}, Accuracy: {test_acc:.4f}")

<>:2: SyntaxWarning: invalid escape sequence '\ '
<>:2: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_22039/3217079108.py:2: SyntaxWarning: invalid escape sequence '\ '
  print(f"\nFinal test results:\nLoss: {test_loss:.4f}\ Accuracy: {test_acc:.4f}")
Evaluating: 100%|██████████| 238/238 [00:04<00:00, 58.59it/s, loss=0.4990, acc=0.9007]


Final test results:
Loss: 0.2967, Accuracy: 0.9007


# Testing the results

In [55]:
def predict(model, tokenizer, text, device):
  class_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

  model.eval()
  with torch.no_grad():
    encoded = tokenizer.encode(text)
    inputs = torch.tensor([encoded.ids], dtype=torch.long).to(device)

    outputs = model(inputs)
    _, predicted = torch.max(outputs, dim=1)

    class_idx = predicted.item()
    return class_map.get(class_idx, "Unknown")

sample_text = "The new smartphone features a powerful processor and advanced camera system."
result = predict(model, tokenizer, sample_text, device)
print(f"Text: {sample_text}")
print(f"Predicted Class: {result}")

Text: The new smartphone features a powerful processor and advanced camera system.
Predicted Class: Sci/Tech
